In [1]:
# Cell 1: Install Required Libraries
!pip install -q transformers torch accelerate sentencepiece

In [2]:
# Cell 2: Import All Required Libraries

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import json
import random
import time
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np
import statistics

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [3]:
# Cell 3: Upload Data Files
# Upload: noun_universe.txt, all_per_student.json

from google.colab import files

print("📁 Please upload your data files:")
print("   1. noun_universe.txt")
print("   2. all_per_student.json")
print("\nClick 'Choose Files' below...")

uploaded = files.upload()

print("\n✅ Files uploaded successfully!")
print(f"Uploaded files: {list(uploaded.keys())}")

📁 Please upload your data files:
   1. noun_universe.txt
   2. all_per_student.json

Click 'Choose Files' below...


Saving all_per_student.json to all_per_student.json
Saving noun_universe.txt to noun_universe.txt

✅ Files uploaded successfully!
Uploaded files: ['all_per_student.json', 'noun_universe.txt']


In [4]:
# Cell 4: Load All Nouns from the Universe

# Load noun universe
with open('noun_universe.txt', 'r') as f:
    all_nouns = [line.strip() for line in f.readlines() if line.strip()]

print(f"✅ Loaded {len(all_nouns)} nouns from noun_universe.txt")
print(f"\n📊 First 20 nouns:")
for i, noun in enumerate(all_nouns[:20], 1):
    print(f"   {i}. {noun}")

print(f"\n📊 Last 10 nouns:")
for i, noun in enumerate(all_nouns[-10:], len(all_nouns)-9):
    print(f"   {i}. {noun}")

✅ Loaded 1852 nouns from noun_universe.txt

📊 First 20 nouns:
   1. aardvark
   2. acai
   3. action figure
   4. actor
   5. adobe
   6. advanced micro
   7. aerodactyl
   8. aeronautical engineer
   9. african elephant
   10. africatwin
   11. agate
   12. agility ladder drills
   13. air
   14. air start unit
   15. air traffic controller
   16. aircraft mechanic
   17. airedale terrier
   18. airliner
   19. airplane
   20. airport

📊 Last 10 nouns:
   1843. yard
   1844. yoga
   1845. yogurt
   1846. zapdos
   1847. zebra
   1848. zercher squat
   1849. zircon
   1850. zoo
   1851. zx10r
   1852. zx6r


In [5]:
# Cell 5: Load and Analyze all_per_student.json file

# Load all student data
with open('all_per_student.json', 'r') as f:
    student_data = json.load(f)

print(f"✅ Loaded data from {len(student_data)} students")

# Collect all properties from all students
all_properties = set()
property_frequency = Counter()

for student_id, data in student_data.items():
    if 'properties' in data:
        all_properties.update(data['properties'])
        property_frequency.update(data['properties'])

print(f"\n📊 Total unique properties: {len(all_properties)}")
print(f"\n📊 Top 30 most common properties:")

# Sort by frequency
sorted_properties = property_frequency.most_common(30)
for i, (prop, count) in enumerate(sorted_properties, 1):
    print(f"   {i:2d}. {prop:30s} (used by {count} students)")

✅ Loaded data from 72 students

📊 Total unique properties: 255

📊 Top 30 most common properties:
    1. size                           (used by 40 students)
    2. weight                         (used by 32 students)
    3. speed                          (used by 25 students)
    4. durability                     (used by 13 students)
    5. complexity                     (used by 12 students)
    6. intelligence                   (used by 12 students)
    7. strength                       (used by 10 students)
    8. danger                         (used by 8 students)
    9. aggressiveness                 (used by 7 students)
   10. usefulness                     (used by 7 students)
   11. temperature                    (used by 6 students)
   12. hardness                       (used by 6 students)
   13. value                          (used by 6 students)
   14. rarity                         (used by 5 students)
   15. portability                    (used by 5 students)
   16. popu

In [6]:
# Cell 6: Analyze Property Usefulness for 20 Questions

print("🎯 CRITERIA FOR SELECTING GOOD PROPERTIES:")
print("=" * 70)

criteria = """
1. HIGH DISCRIMINATING POWER
   - Property should split nouns into different groups
   - Example: 'size' (mouse=1, elephant=10) ✓
   - Bad example: 'existence' (everything exists) ✗

2. BROAD APPLICABILITY
   - Property should apply to MOST nouns in universe
   - Example: 'size' applies to all physical objects ✓
   - Bad example: 'horsepower' only applies to vehicles ✗

3. OBJECTIVE & CONSISTENT
   - Different people should rate similarly
   - Example: 'weight' is objective ✓
   - Bad example: 'beauty' is highly subjective ✗

4. BINARY DISTINGUISHABLE
   - Should clearly separate YES/NO answers
   - Example: 'living' (plant=yes, rock=no) ✓
   - Bad example: properties with fuzzy boundaries ✗

5. COMPLEMENTARY SET
   - Properties should work together to narrow down nouns
   - Cover different dimensions: physical, functional, categorical
"""

print(criteria)

print("\n🎯 RECOMMENDED PROPERTY CATEGORIES:")
print("=" * 70)

recommendations = {
    "Physical Dimensions": [
        "size/size_scale_1_10",
        "weight/weight_scale_1_10",
        "height"
    ],
    "Categorical": [
        "natural vs artificial",
        "living vs non-living",
        "edibility/edibleness"
    ],
    "Functional": [
        "utility/usefulness",
        "portability/portability_1_10",
        "mobility"
    ],
    "Abstract": [
        "complexity",
        "popularity",
        "rarity/rarity_1_10"
    ],
    "Environmental": [
        "typical_indoor_use_1_10",
        "temperature",
        "dangerous/danger_to_humans"
    ]
}

for category, props in recommendations.items():
    print(f"\n{category}:")
    for prop in props:
        print(f"   • {prop}")

🎯 CRITERIA FOR SELECTING GOOD PROPERTIES:

1. HIGH DISCRIMINATING POWER
   - Property should split nouns into different groups
   - Example: 'size' (mouse=1, elephant=10) ✓
   - Bad example: 'existence' (everything exists) ✗

2. BROAD APPLICABILITY
   - Property should apply to MOST nouns in universe
   - Example: 'size' applies to all physical objects ✓
   - Bad example: 'horsepower' only applies to vehicles ✗

3. OBJECTIVE & CONSISTENT
   - Different people should rate similarly
   - Example: 'weight' is objective ✓
   - Bad example: 'beauty' is highly subjective ✗

4. BINARY DISTINGUISHABLE
   - Should clearly separate YES/NO answers
   - Example: 'living' (plant=yes, rock=no) ✓
   - Bad example: properties with fuzzy boundaries ✗

5. COMPLEMENTARY SET
   - Properties should work together to narrow down nouns
   - Cover different dimensions: physical, functional, categorical


🎯 RECOMMENDED PROPERTY CATEGORIES:

Physical Dimensions:
   • size/size_scale_1_10
   • weight/weight_scale

In [7]:
# Cell 7: Define Your Selected Properties
# EDIT THIS - Choose 5-10 properties based on the analysis above

# Strategy: Choose properties that cover different aspects
# and are most commonly available in student data

SELECTED_PROPERTIES = [
    # Physical properties (splits by size/scale)
    "size",
    "weight",

    # Categorical properties (broad categories)
    "speed",  # or use as a property like "fast vs slow"
    "durability",  # or "fragile vs durable"

    # Functional properties
    "complexity",
    "intelligence",

    # Danger/Safety (helps distinguish animals, weapons, etc.)
    "danger",  # or "dangerousness"

    # Abstract properties
    "usefulness",
    "value",

    # Environmental/Sensory
    "temperature",  # hot vs cold items
]

print(f"✅ Selected {len(SELECTED_PROPERTIES)} properties:")
for i, prop in enumerate(SELECTED_PROPERTIES, 1):
    print(f"  {i}. {prop}")

# Verify these properties exist in student data
available_count = 0
for prop in SELECTED_PROPERTIES:
    count = property_frequency.get(prop, 0)
    available_count += (count > 0)
    status = "✓" if count > 0 else "✗ NOT FOUND"
    print(f"  {prop:30s} - {count:3d} students {status}")

print(f"\n✅ {available_count}/{len(SELECTED_PROPERTIES)} properties found in student data")

if available_count < len(SELECTED_PROPERTIES):
    print("\n⚠️  WARNING: Some properties not found!")
    print("    You may need to adjust your property list or generate those ratings yourself (next steps)")

✅ Selected 10 properties:
  1. size
  2. weight
  3. speed
  4. durability
  5. complexity
  6. intelligence
  7. danger
  8. usefulness
  9. value
  10. temperature
  size                           -  40 students ✓
  weight                         -  32 students ✓
  speed                          -  25 students ✓
  durability                     -  13 students ✓
  complexity                     -  12 students ✓
  intelligence                   -  12 students ✓
  danger                         -   8 students ✓
  usefulness                     -   7 students ✓
  value                          -   6 students ✓
  temperature                    -   6 students ✓

✅ 10/10 properties found in student data


In [8]:
# Cell 8: Extract Prompts for Your Selected Properties

# Build a mapping of property -> prompts from student data
property_prompts = {}

for student_id, data in student_data.items():
    if 'prompts' in data:
        prompts_data = data['prompts']

        # Handle different possible structures
        if isinstance(prompts_data, dict):
            # If prompts is a dictionary: {property: prompt}
            for prop_name, prompt_text in prompts_data.items():
                if prop_name in SELECTED_PROPERTIES:
                    if prop_name not in property_prompts:
                        property_prompts[prop_name] = []
                    property_prompts[prop_name].append({
                        'student': student_id,
                        'pre-prompt': prompt_text
                    })

        elif isinstance(prompts_data, list):
            # If prompts is a list
            for prompt_obj in prompts_data:
                if isinstance(prompt_obj, dict):
                    # List of dictionaries: [{'property': 'size', 'pre-prompt': '...'}]
                    prop_name = prompt_obj.get('property', '')
                    if prop_name in SELECTED_PROPERTIES:
                        if prop_name not in property_prompts:
                            property_prompts[prop_name] = []
                        property_prompts[prop_name].append({
                            'student': student_id,
                            'pre-prompt': prompt_obj.get('pre-prompt', '')
                        })
                elif isinstance(prompt_obj, str):
                    # List of strings - skip or handle differently
                    continue

print(f"📋 Found prompts for selected properties:")
print("=" * 70)

for prop in SELECTED_PROPERTIES:
    if prop in property_prompts:
        print(f"\n✅ {prop}: {len(property_prompts[prop])} prompts available")
        # Show first example
        example = property_prompts[prop][0]
        print(f"   Example: {example['pre-prompt'][:100]}...")
    else:
        print(f"\n❌ {prop}: NO prompts found - NEED TO CREATE")

# Properties we need to create prompts for
missing_properties = [p for p in SELECTED_PROPERTIES if p not in property_prompts]

if missing_properties:
    print(f"\n⚠️  Need to create prompts for: {missing_properties}")
else:
    print(f"\n✅ All properties have existing prompts!")

📋 Found prompts for selected properties:

✅ size: 22 prompts available
   Example: You are a strict rater that assigns a single integer score.
Property: size
Anchors: 1=tiny (mouse), ...

✅ weight: 20 prompts available
   Example: Think about all of the many useful objects in someone's daily life. Some are heavier than others, an...

✅ speed: 17 prompts available
   Example: You are a strict rater that assigns a single integer score.
Property: speed
Anchors: 1=very slow (tu...

✅ durability: 9 prompts available
   Example: Evaluate the Pokemon's defensive toughness, stamina, and resistances from pokemon games, pokedex, ca...

✅ complexity: 8 prompts available
   Example: You are a TV show analyst who responds with a single integer from 1 to 10 and nothing else. Do not a...

✅ intelligence: 9 prompts available
   Example: You are a strict rater that assigns a single integer score.
Property: intelligence
Anchors: 1=simple...

✅ danger: 5 prompts available
   Example: Rate how dangerous t

In [9]:
# Cell 9: Create Final Standardized Prompts for ALL Properties

# For each property, create ONE standardized prompt
# We'll either use the best student prompt or create our own

FINAL_PROMPTS = {}

# Template for creating new prompts
def create_prompt_for_property(property_name):
    """
    Create a standardized prompt for a property
    Returns a prompt that asks for 1-10 rating
    """

    prompt_templates = {
        "size": "Rate from 1-10 only, no explanation. Physical size of this object (1=tiny like ant, 10=huge like whale): ",
        "weight": "Rate from 1-10 only, no explanation. Weight of this object (1=very light like feather, 10=very heavy like elephant): ",
        "living": "Rate from 1-10 only, no explanation. Is this alive? (1=definitely not alive like rock, 10=definitely alive like human): ",
        "artificial": "Rate from 1-10 only, no explanation. Is this man-made/artificial? (1=completely natural like tree, 10=completely artificial like computer): ",
        "utility": "Rate from 1-10 only, no explanation. How useful is this in daily life? (1=not useful/decorative, 10=essential for survival): ",
        "portability": "Rate from 1-10 only, no explanation. How portable/easy to carry? (1=impossible to move like mountain, 10=fits in pocket): ",
        "dangerous": "Rate from 1-10 only, no explanation. How dangerous to humans? (1=completely safe like pillow, 10=extremely dangerous like lion): ",
        "complexity": "Rate from 1-10 only, no explanation. How complex? (1=very simple like stick, 10=very complex like computer): ",
        "popularity": "Rate from 1-10 only, no explanation. How well-known/popular? (1=almost unknown, 10=universally known): ",
        "typical_indoor_use_1_10": "Rate from 1-10 only, no explanation. Typically used indoors? (1=always outdoor like tree, 10=always indoor like bed): "
    }

    return prompt_templates.get(property_name,
        f"Rate from 1-10 only, no explanation. {property_name} of this: ")

# Build final prompts - use best student prompt or create new
for prop in SELECTED_PROPERTIES:
    if prop in property_prompts and len(property_prompts[prop]) > 0:
        # Use most common student prompt (or first one)
        FINAL_PROMPTS[prop] = property_prompts[prop][0]['pre-prompt']
    else:
        # Create our own
        FINAL_PROMPTS[prop] = create_prompt_for_property(prop)

print("✅ Final Standardized Prompts:")
print("=" * 70)

for i, (prop, prompt) in enumerate(FINAL_PROMPTS.items(), 1):
    print(f"\n{i}. Property: {prop}")
    print(f"   Prompt: {prompt}")

# Save prompts to file for reference
with open('final_prompts.json', 'w') as f:
    json.dump(FINAL_PROMPTS, f, indent=2)

print("\n✅ Prompts saved to 'final_prompts.json'")

✅ Final Standardized Prompts:

1. Property: size
   Prompt: You are a strict rater that assigns a single integer score.
Property: size
Anchors: 1=tiny (mouse), 10=huge (whale).
Guidelines:
 - Consider a typical adult specimen in the wild (common interpretation).
 - If uncertain, make your best estimate; do not answer with text.
 - Output ONLY one integer from 1 to 10 (no words, no punctuation, no units).
Question: What is the rating for this animal?
Noun: 

2. Property: weight
   Prompt: Think about all of the many useful objects in someone's daily life. Some are heavier than others, and some are lighter. Let's say you were also told to rate such heaviness from 1 to 10. For example, refrigerator: 10, bicycle: 8, mirror: 5, vase: 4, watch: 2, pen: 1. Think not just of these examples, but those inbetween! Also, keep in mind that power and weight are NOT the same! Now, I will give you an item, and I want you to rate it on a scale from 1 to 10. DO NOT SAY ANYTHING OTHER THAN A NUMBER FROM 

In [10]:
# Cell 10: Load Phi-3-mini-4k-instruct Model

print("🤖 Loading Phi-3-mini-4k-instruct model...")
print("⏱️  This may take 2-5 minutes on first run...")

model_name = "microsoft/Phi-3-mini-4k-instruct"

try:
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )

    # Load model with memory-efficient settings
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16,  # Use float16 for memory efficiency
        trust_remote_code=True
    )

    # Create text generation pipeline
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
    )

    print("✅ Model loaded successfully!")
    print(f"   Model: {model_name}")
    print(f"   Device: {model.device}")

except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("\nIf you get memory errors, try:")
    print("   Runtime -> Change runtime type -> T4 GPU")

🤖 Loading Phi-3-mini-4k-instruct model...
⏱️  This may take 2-5 minutes on first run...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Device set to use cuda:0


✅ Model loaded successfully!
   Model: microsoft/Phi-3-mini-4k-instruct
   Device: cuda:0
